# Visualizing the kilonova transformer's attention

The analogue of tensor2tensor's `hello_t2t` notebook for `KilonovaTransformer`. The model reads a
short sequence of light-curve tokens and prepends two global tokens:

```
[CLS]  [Z]  R·d +0d   Z·d +0d   ...   H·u +5d   ...
```

Classification reads only the `CLS` output, so **the CLS row of the attention matrix is literally
"what the classifier looks at"** — which band / epoch / token-type the decision leans on. This
notebook pulls the per-layer attention with `model.attention_maps(batch)` and renders it two ways:

1. a layer x head grid of attention heatmaps over the labelled token sequence, and
2. the CLS attention weights laid over the example's light curve.

**Data / weights.** The real run wants the trained checkpoint (`kilonova_transformer-soup.ckpt`) and
the full OpenUniverse token cache — both live on the training laptop. When neither is present the
setup falls back to the KN-window hdf5 in `data/openuniverse/` and an **untrained** model, so the
plotting machinery still runs end to end (the attention is then meaningless — random weights).

In [ ]:
import os
import sys

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import gridspec

sys.path.insert(0, os.path.abspath('.'))  # training/ modules: model, openuniverse_data, train_lightning
import openuniverse_data as oud
from model import BAND_ORDER, TOKEN_TYPE_ORDER, KilonovaTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'


def first_existing(candidates, default):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return default


# where the real assets live when run on the training laptop; both optional here.
CHECKPOINT = first_existing(
    ['checkpoints/kilonova_transformer-soup.ckpt'],
    'checkpoints/kilonova_transformer-soup.ckpt')
DATA_DIR = first_existing(['data/openuniverse', '../data/openuniverse'], 'data/openuniverse')
TOKEN_CACHE = f'{DATA_DIR}/openuniverse_tokens.npz'

C_KN = '#FB5607'      # detections / the KN class
C_UPPER = '#3A86FF'   # upper limits
C_ABSENT = '#8D99AE'  # not-observed tokens
C_GLOBAL = '#2EC4B6'  # CLS / [Z] global tokens

TYPE_COLOR = {'d': C_KN, 'u': C_UPPER, 'n': C_ABSENT}
TYPE_NAME = {'d': 'detection', 'u': 'upper limit', 'n': 'not observed'}
print('device:', device, '| DATA_DIR:', DATA_DIR)


In [ ]:
def load_model():
    """Trained model from the checkpoint when available, else an untrained KilonovaTransformer.
    Returns (model_in_eval_mode, is_trained)."""
    if os.path.exists(CHECKPOINT):
        from train_lightning import LitKilonova
        from openuniverse_data import GROUP_ORDER
        lit = LitKilonova.load_from_checkpoint(
            CHECKPOINT, class_weights=torch.ones(len(GROUP_ORDER)))
        model = lit.model.to(device).eval()
        return model, True
    print('checkpoint not found -> UNTRAINED model (attention is random, machinery only)')
    return KilonovaTransformer().to(device).eval(), False


model, is_trained = load_model()
n_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f'parameters: {n_parameters:,}  |  trained: {is_trained}')

In [ ]:
def build_example_batch(n_examples=8):
    """A small collated batch to visualize. Preferred path: the real leakage-aware test split from
    the token cache (mixed KN + contaminants). Fallback: the KN-window hdf5 shipped in the repo
    (KN only). Sets the openuniverse_data magnitude-normalization globals either way."""
    if os.path.exists(TOKEN_CACHE):
        from openuniverse_data import (_load_or_build, _leakage_aware_split,
                                        OpenUniverseWindowDataset, collate_token_windows,
                                        GROUP_TO_LABEL)
        big, meta = _load_or_build(TOKEN_CACHE, None, None, None, None, verbose=False)
        label_by_index = np.where(meta['is_kn'], GROUP_TO_LABEL['KN'], GROUP_TO_LABEL['other'])
        _, _, test_index = _leakage_aware_split(meta, (0.90, 0.05, 0.05), 42)
        source = 'cache'
    else:
        print(f'{TOKEN_CACHE} not found -> KN-window hdf5 fallback (KN only)')
        big, counts, meta = oud._read_kn_hdf5(f'{DATA_DIR}/kilonova_windows_deep.hdf5',
                                              'deep', verbose=False)
        offsets = np.zeros(len(counts) + 1, dtype=np.int64)
        np.cumsum(counts, out=offsets[1:])
        meta['offsets'] = offsets
        label_by_index = np.ones(len(counts), dtype=np.int64)
        test_index = np.arange(len(counts))
        source = 'kn_hdf5'

    detections = big['token_type_index'] == oud.TOKEN_TYPE_TO_INDEX['d']
    oud.MAG_MEAN = float(np.nanmean(big['mag'][detections]))
    oud.MAG_STD = float(np.nanstd(big['mag'][detections]))
    oud.SIGMA_MAG_MEAN = float(np.nanmean(big['sigma_mag'][detections]))
    oud.SIGMA_MAG_STD = float(np.nanstd(big['sigma_mag'][detections]))

    from openuniverse_data import OpenUniverseWindowDataset, collate_token_windows
    dataset = OpenUniverseWindowDataset(
        test_index[:n_examples], big=big, meta=meta, label_by_index=label_by_index,
        data_aug=False, force_epochs=3, force_redshift=True)
    batch = collate_token_windows([dataset[i] for i in range(len(dataset))])
    return batch, source


batch, batch_source = build_example_batch()
print('source:', batch_source, '| examples:', batch['label'].shape[0],
      '| padded tokens:', batch['delta_time'].shape[1])

In [ ]:
def sequence_token_labels(batch, example_index):
    """Human-readable label per position of the model's input sequence for one example:
    [CLS, [Z]/[noZ], then one 'band type +dt' per real light-curve token] (padding dropped).
    Returns (labels, token_types) where token_types aligns with labels (globals -> 'g')."""
    valid = ~batch['padding_mask'][example_index]
    n_tokens = int(valid.sum().item())
    has_z = batch['has_redshift'][example_index].item() > 0.5
    labels = ['CLS', '[Z]' if has_z else '[noZ]']
    token_types = ['g', 'g']
    for position in range(n_tokens):
        band = BAND_ORDER[int(batch['band_index'][example_index, position])]
        token_type = TOKEN_TYPE_ORDER[int(batch['token_type_index'][example_index, position])]
        delta_time = float(batch['delta_time'][example_index, position])
        labels.append(f'{band} {token_type} {delta_time:+.0f}d')
        token_types.append(token_type)
    return labels, token_types


labels_preview, _ = sequence_token_labels(batch, 0)
print('example 0 sequence:', labels_preview)

In [ ]:
def plot_attention_grid(model, batch, example_index, plot=True):
    """Full layer x head attention grid for one example. Each heatmap is the softmax attention over
    the valid sequence (globals + real tokens), rows = query position, cols = key position."""
    with torch.no_grad():
        attention_maps = model.attention_maps({k: v.to(device) for k, v in batch.items()})
    labels, _ = sequence_token_labels(batch, example_index)
    length = len(labels)
    num_layers = len(attention_maps)
    num_heads = attention_maps[0].shape[1]

    if not plot:
        return attention_maps

    fig, axes = plt.subplots(num_layers, num_heads,
                             figsize=(3.1 * num_heads, 3.1 * num_layers), squeeze=False)
    for layer in range(num_layers):
        for head in range(num_heads):
            weights = attention_maps[layer][example_index, head, :length, :length].cpu().numpy()
            ax = axes[layer][head]
            image = ax.imshow(weights, cmap='magma', vmin=0.0, aspect='equal')
            ax.set_xticks(range(length))
            ax.set_yticks(range(length))
            ax.set_xticklabels(labels, rotation=90, fontsize=7)
            ax.set_yticklabels(labels, fontsize=7)
            ax.set_title(f'layer {layer} · head {head}', fontsize=10)
            fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    axes[num_layers - 1][0].set_xlabel('key (attended to)')
    axes[0][0].set_ylabel('query')
    fig.suptitle(f'Attention · example {example_index} ({group_label(batch, example_index)})',
                 y=1.01, fontsize=13)
    fig.tight_layout()
    plt.show()
    return attention_maps


def group_label(batch, example_index):
    return 'KN' if int(batch['label'][example_index]) == 1 else 'other'


def plot_cls_attention_on_lightcurve(model, batch, example_index, layer=-1, plot=True):
    """The CLS attention row (mean over heads) for one example, shown two ways: a bar over the
    labelled sequence, and the same weights sized/coloured on the light curve (mag vs dt). This is
    'what the classifier looks at'. Only detection/upper-limit tokens carry photometry, so
    not-observed ('n') tokens appear in the bar chart but not on the light curve."""
    with torch.no_grad():
        attention_maps = model.attention_maps({k: v.to(device) for k, v in batch.items()})
    labels, token_types = sequence_token_labels(batch, example_index)
    length = len(labels)
    cls_attention = attention_maps[layer][example_index, :, 0, :length].mean(dim=0).cpu().numpy()

    if not plot:
        return cls_attention

    n_tokens = int((~batch['padding_mask'][example_index]).sum().item())
    delta_time = batch['delta_time'][example_index, :n_tokens].cpu().numpy()
    band_index = batch['band_index'][example_index, :n_tokens].cpu().numpy()
    type_index = batch['token_type_index'][example_index, :n_tokens].cpu().numpy()
    magnitude = batch['magnitude'][example_index, :n_tokens].cpu().numpy() * oud.MAG_STD + oud.MAG_MEAN
    token_weights = cls_attention[2:]  # positions align with token 0..n_tokens-1
    observed = type_index != oud.TOKEN_TYPE_TO_INDEX['n']  # only 'd'/'u' have a magnitude

    fig = plt.figure(figsize=(14, 4.6))
    grid = gridspec.GridSpec(1, 2, width_ratios=[1.15, 1.0], wspace=0.28)

    ax_bar = fig.add_subplot(grid[0])
    colors = [C_GLOBAL if token_type == 'g' else TYPE_COLOR[token_type] for token_type in token_types]
    ax_bar.bar(range(length), cls_attention, color=colors)
    ax_bar.set_xticks(range(length))
    ax_bar.set_xticklabels(labels, rotation=90, fontsize=8)
    ax_bar.set_ylabel('CLS attention weight')
    ax_bar.set_title(f'CLS attention (layer {layer}, mean over heads)')
    ax_bar.grid(axis='y', alpha=0.3)

    ax_lc = fig.add_subplot(grid[1])
    scale = 1600.0 / max(token_weights[observed].max(), 1e-6) if observed.any() else 1.0
    for band_position in range(len(BAND_ORDER)):
        in_band = observed & (band_index == band_position)
        if in_band.sum() >= 2:
            order = np.argsort(delta_time[in_band])
            ax_lc.plot(delta_time[in_band][order], magnitude[in_band][order],
                       '-', color=C_ABSENT, alpha=0.4, zorder=1)
    for token_type_name in ('d', 'u'):
        is_type = observed & (type_index == oud.TOKEN_TYPE_TO_INDEX[token_type_name])
        if not is_type.any():
            continue
        ax_lc.scatter(delta_time[is_type], magnitude[is_type],
                      s=token_weights[is_type] * scale + 12, color=TYPE_COLOR[token_type_name],
                      edgecolor='black', linewidth=0.5, label=TYPE_NAME[token_type_name], zorder=2)
    ax_lc.invert_yaxis()
    ax_lc.set_xlabel(r'$\Delta t$ since first detection [d]')
    ax_lc.set_ylabel('AB magnitude')
    ax_lc.set_title('marker size = CLS attention')
    ax_lc.legend(fontsize=8, loc='best')
    ax_lc.grid(alpha=0.3)

    fig.suptitle(f'What the classifier looks at · example {example_index} '
                 f'({group_label(batch, example_index)})', y=1.03, fontsize=13)
    plt.show()
    return cls_attention


## Full attention grid

Every layer x head attention matrix for one example. Query on the y-axis attends to keys on the
x-axis; row 0 is the `CLS` token whose output feeds the classifier.

In [ ]:
_ = plot_attention_grid(model, batch, example_index=0, plot=True)

## What the classifier looks at

The `CLS` attention row (averaged over heads, last layer) as a bar over the token sequence and as
marker size on the light curve. Colour = token type (detection / upper limit / not observed).

In [ ]:
_ = plot_cls_attention_on_lightcurve(model, batch, example_index=0, layer=-1, plot=True)

## A second example

Swap `example_index` to compare objects (e.g. a KN vs. a contaminant when running off the full
token cache).

In [ ]:
_ = plot_cls_attention_on_lightcurve(model, batch, example_index=1, layer=-1, plot=True)